In [1]:
import sqlite3
import requests
from bs4 import BeautifulSoup


In [2]:
url="https://prsindia.org/billtrack"
header={
    "User-Agent":"Mozilla/5.0"
}

In [3]:

# 1. Setup the Database
def setup_db():
    conn = sqlite3.connect('bills_data.db')
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS bills (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            heading TEXT,
            link TEXT
        )
    ''')
    conn.commit()
    return conn, cursor

# 2. Scrape the Content
def scrape_and_store(url, conn, cursor):
    # Note: Replace 'url' with the actual page URL or use a local HTML file
    # For testing with the image content:
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')

    # Target the 'views-row' containers seen in your source code
    rows = soup.find_all('div', class_='views-row')

    for row in rows:
        # Find the h3 tag within the row
        h3_tag = row.find('h3', class_='cate')
        if h3_tag:
            a_tag = h3_tag.find('a')
            if a_tag:
                heading = a_tag.get_text(strip=True)
                link = a_tag.get('href')

                # Insert into SQL
                cursor.execute('INSERT INTO bills (heading, link) VALUES (?, ?)', (heading, link))
                print(f"Stored: {heading}")

    conn.commit()

# Main Execution
if __name__ == "__main__":
    target_url = "https://prsindia.org/billtrack" # Update as needed
    db_conn, db_cursor = setup_db()

    try:
        scrape_and_store(target_url, db_conn, db_cursor)
        print("\nScraping completed successfully.")
    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        db_conn.close()


Stored: The Andhra Pradesh Reorganisation (Amendment) Bill, 2026
Stored: The Jan Vishwas (Amendment of Provisions) Bill, 2026
Stored: The Central Armed Police Forces (General Administration) Bill, 2026
Stored: The Foreign Contribution (Regulation) Amendment Bill, 2026
Stored: The Corporate Laws (Amendment) Bill, 2026
Stored: The Transgender Persons (Protection of Rights) Amendment Bill, 2026
Stored: The Industrial Relations Code (Amendment) Bill, 2026
Stored: The Securities Markets Code, 2025
Stored: The Viksit Bharat – Guarantee for Rozgar and Ajeevika Mission (Gramin) VB – G RAM G Bill, 2025
Stored: The Sabka Bima Sabki Raksha (Amendment of Insurance Laws) Bill, 2025
Stored: The Repealing and Amending Bill, 2025
Stored: The Sustainable Harnessing and Advancement of Nuclear Energy for Transforming India Bill, 2025
Stored: The Viksit Bharat Shiksha Adhishthan Bill, 2025
Stored: The Manipur Goods and Services Tax (Second Amendment) Bill, 2025
Stored: The Central Excise (Amendment) Bill,

In [4]:
import pandas as pd
import sqlite3

# Connect to the database you created
conn = sqlite3.connect('bills_data.db')

# Load the SQL table into a Pandas DataFrame
df = pd.read_sql_query("SELECT * FROM bills", conn)

# Display the data as a clean table
conn.close()
df


,id,heading,link,pdf_url
0,1,The Jan Vishwas (Amendment of Provisions) Bill...,/billtrack/the-jan-vishwas-amendment-of-provis...,NaN
1,2,The Central Armed Police Forces (General Admin...,/billtrack/the-central-armed-police-forces-gen...,NaN
2,3,The Foreign Contribution (Regulation) Amendmen...,/billtrack/the-foreign-contribution-regulation...,NaN
3,4,"The Corporate Laws (Amendment) Bill, 2026",/billtrack/the-corporate-laws-amendment-bill-2026,NaN
4,5,The Transgender Persons (Protection of Rights)...,/billtrack/the-transgender-persons-protection-...,NaN
...,...,...,...,...
1904,1905,"The Lok Pal Bill, 2011",/billtrack/the-lok-pal-bill-2011,NaN
1905,1906,Earlier Lok Pal Bills,/billtrack/earlier-lok-pal-bills,NaN
1906,1907,The Land Acquisition and Resettlement and Reha...,/billtrack/the-land-acquisition-and-resettleme...,NaN
1907,1908,"The Jan Lok Pal Bill, 2011",/billtrack/the-jan-lok-pal-bill-2011,NaN


In [ ]:

import time

# 1. Connect to your existing database
db_name = 'bills_data.db' # Ensure this matches your filename
conn = sqlite3.connect(db_name)
cursor = conn.cursor()

# 2. Add the pdf_url column if it doesn't exist yet
try:
    cursor.execute('ALTER TABLE bills ADD COLUMN pdf_url TEXT')
    conn.commit()
except sqlite3.OperationalError:
    pass # Column already exists

# 3. Fetch all existing links to process
cursor.execute("SELECT id, link FROM bills")
rows = cursor.fetchall()

print(f"Found {len(rows)} entries to process...")

for row_id, page_url in rows:
    try:
        # Construct full URL if needed
        full_url = f"https://prsindia.org{page_url}" if page_url.startswith('/') else page_url

        # Request the page
        response = requests.get(full_url, timeout=10)
        soup = BeautifulSoup(response.text, 'html.parser')

        # Target the <ul> and <a> tags from your second screenshot
        pdf_section = soup.find('ul', class_='pdf_html_links')
        if pdf_section:
            a_tag = pdf_section.find('a', class_='pdf-link')
            if a_tag:
                pdf_link = a_tag.get('href')

                # Update the specific row in SQL
                cursor.execute("UPDATE bills SET pdf_url = ? WHERE id = ?", (pdf_link, row_id))
                print(f"Updated ID {row_id} with PDF: {pdf_link}")

        conn.commit()
        time.sleep(1) # Delay to prevent IP blocking

    except Exception as e:
        print(f"Error on ID {row_id}: {e}")

conn.close()
print("\nUpdate complete.")


Found 1909 entries to process...
